## Regression with an Abalone Dataset

This project predicts the age of abalone (a type of sea snail) from physical measurements. The age is given by the number of shell rings, which normally requires cutting the shell and counting under a microscope, so predicting it from simple measurements saves time.

## Approach
1. Load the data and explore the target and features
2. Encode the categorical feature (sex)
3. Log-transform the target to match the RMSLE metric
4. Train a baseline regression model and evaluate it on a validation split
5. Generate predictions and create the submission file
6. Submit to Kaggle and record the score

In [1]:
import os
from getpass import getpass

os.environ["KAGGLE_API_TOKEN"] = getpass("Paste your Kaggle API token and press Enter: ")

!pip install -q -U kaggle
!kaggle competitions download -c playground-series-s4e4
!unzip -oq playground-series-s4e4.zip

Paste your Kaggle API token and press Enter: ··········
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.2/126.2 kB 3.7 MB/s eta 0:00:00
100% 2.41M/2.41M [00:00<00:00, 184MB/s]



In [2]:
import pandas as pd
import numpy as np

train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

In [3]:
print(train.shape, test.shape)

(90615, 10) (60411, 9)


In [4]:
train.sample(1)

,id,Sex,Length,Diameter,Height,Whole weight,Whole weight.1,Whole weight.2,Shell weight,Rings
59159,59159,I,0.43,0.32,0.115,0.3485,0.1405,0.066,0.11,8


In [5]:
test.sample(1)

,id,Sex,Length,Diameter,Height,Whole weight,Whole weight.1,Whole weight.2,Shell weight
25828,116443,I,0.255,0.19,0.04,0.081,0.0265,0.015,0.025


In [6]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 90615 entries, 0 to 90614
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   id              90615 non-null  int64  
 1   Sex             90615 non-null  object 
 2   Length          90615 non-null  float64
 3   Diameter        90615 non-null  float64
 4   Height          90615 non-null  float64
 5   Whole weight    90615 non-null  float64
 6   Whole weight.1  90615 non-null  float64
 7   Whole weight.2  90615 non-null  float64
 8   Shell weight    90615 non-null  float64
 9   Rings           90615 non-null  int64  
dtypes: float64(7), int64(2), object(1)
memory usage: 6.9+ MB


In [7]:
test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 60411 entries, 0 to 60410
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   id              60411 non-null  int64  
 1   Sex             60411 non-null  object 
 2   Length          60411 non-null  float64
 3   Diameter        60411 non-null  float64
 4   Height          60411 non-null  float64
 5   Whole weight    60411 non-null  float64
 6   Whole weight.1  60411 non-null  float64
 7   Whole weight.2  60411 non-null  float64
 8   Shell weight    60411 non-null  float64
dtypes: float64(7), int64(1), object(1)
memory usage: 4.1+ MB


In [8]:
train.columns.tolist()

['id',
 'Sex',
 'Length',
 'Diameter',
 'Height',
 'Whole weight',
 'Whole weight.1',
 'Whole weight.2',
 'Shell weight',
 'Rings']

In [9]:
train["Rings"].describe()

,Rings
count,90615.000000
mean,9.696794
std,3.176221
min,1.000000
25%,8.000000
50%,9.000000
75%,11.000000
max,29.000000


In [10]:
test_ids = test["id"]
train = train.drop(columns=["id"])
test = test.drop(columns=["id"])

In [11]:
train = pd.get_dummies(train, columns=["Sex"])

In [12]:
test = pd.get_dummies(test, columns=["Sex"])

In [13]:
train, test = train.align(test, join="left", axis=1, fill_value=0)

In [14]:
train.shape, test.shape

((90615, 11), (60411, 11))

In [16]:
x = train.drop(columns=["Rings"])
y = np.log1p(train["Rings"])

In [17]:
x.shape, y.shape

((90615, 10), (90615,))

In [18]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

x_train, x_val, y_train, y_val = train_test_split(x, y, test_size=0.2, random_state=42)

model = RandomForestRegressor(n_estimators=150, max_depth=12, min_samples_leaf=5, random_state=42)
model.fit(x_train, y_train)

pred = model.predict(x_val)
rmsle = mean_squared_error(y_val, pred) ** 0.5
print("Validation RMSLE:", round(rmsle, 4))

Validation RMSLE: 0.152


In [19]:
model.fit(x, y)

x_test = test.drop(columns=["Rings"])
pred_test = np.expm1(model.predict(x_test))

pd.DataFrame({"id": test_ids, "Rings": pred_test}).to_csv("submission.csv", index=False)

In [20]:
!kaggle competitions submit -c playground-series-s4e4 -f submission.csv -m "RandomForest baseline"

100% 1.45M/1.45M [00:00<00:00, 4.09MB/s]
99 submissions remaining today.
Successfully submitted to Regression with an Abalone Dataset

In [21]:
model = RandomForestRegressor(n_estimators=100, max_depth=10, min_samples_leaf=10, random_state=42)
model.fit(x, y)

import pickle, os
pickle.dump(model, open("abalone_model.pkl", "wb"))
print("size MB:", round(os.path.getsize("abalone_model.pkl") / 1e6, 1))

size MB: 9.7
